In [ ]:
import spacy
from spacy.tokens import Span
import pyab3p
import os
import re
import pandas as pd
import json

path_to_custom_model = os.path.normpath('C:\\Users\\jace\\Documents\\assignments\\dissertation DLC content\\custom_ner')

scispacy_model = spacy.load(path_to_custom_model)

In [ ]:
filename = 'pmcA2575403'

with open(f'..\\..\\dissertation DLC content\\manual-corpus-species-1.0\\txt\\{filename}.txt', 'r') as f:
    full_paper_text = f.read()

tags = pd.read_csv('..\\..\\dissertation DLC content\\manual-corpus-species-1.0\\filtered_tags.tsv', sep='\t')

with open('list_files\\ncbi_ids.json', 'r') as f:
    species_ids = json.load(f)

filerows = tags.loc[tags['document'] == filename, ['#entity id', 'start', 'end', 'text']]

ab3p_mapping = {}

abbrev = pyab3p.Ab3p()

abbreviations = abbrev.get_abbrs(full_paper_text)

for abbrev in abbreviations:
    abbrev_ents = scispacy_model(abbrev.long_form)
    if 'MICROBE_NAME' in [ent.label_ for ent in abbrev_ents.ents]:
        ab3p_mapping[abbrev.short_form] = abbrev.long_form

print(ab3p_mapping)

microbes = scispacy_model(full_paper_text)

print(f"entities before: {len([ent.text for ent in microbes.ents if ent.label_ == 'MICROBE_NAME'])}")

print()

for short, long in ab3p_mapping.items():
    matches = re.finditer(short, full_paper_text)
    for match in matches:
        span = microbes.char_span(match.start(), match.end(), label="MICROBE_NAME")
        if span != None:
            #print(span.start, span.end)
            microbes.set_ents([Span(microbes, span.start, span.end, label='MICROBE_NAME')], default='unmodified')

print(f"entities after: {len([ent.text for ent in microbes.ents if ent.label_ == 'MICROBE_NAME'])}")

found_count = 0
total_count = 0
false_positives = 0

all_microbe_names = []

for ind, row in filerows.iterrows():
    try:
        species_id = filerows['#entity id'][ind].split(':ncbi:')[1]
    except:
        continue
    if species_id in species_ids:
        name = row['text'].lower()
        all_microbe_names.append(row['text'])
        total_count += 1

names_found = []

for ent in microbes.ents:
    if not 'MICROBE' in ent.label_:
        continue
    if ent.text not in all_microbe_names:
        continue
    """ try:
        print(filerows['#entity id'])
        species_id = filerows['#entity id'][1].split(':ncbi:')[1]
    except:
        continue
    if not species_id in species_ids:
        continue """
    row_found = filerows[(filerows['text'] == ent.text) & (filerows['start'] == ent.start_char) & (filerows['end'] == ent.end_char)]
    if not row_found.empty:
        found_count += 1
        names_found.append(ent.text)
    else:
        false_positives += 1

{'M.tb': 'Mycobacterium tuberculosis'}
entities before: 51

entities after: 96
